### **Representación multimodal, estrategias de fusión y clasificación multimodal**

Este cuaderno sirve como apoyo teórico-práctico para estudiar una secuencia completa de ideas de fusión multimodal en un problema supervisado con **texto + imagen**. La meta no es alcanzar estado del arte, sino construir una **línea base rigurosa**, entender **qué tipo de interacción entre modalidades se modela** en cada estrategia y dejar preparado el puente hacia modelos más expresivos.

#### **Qué cubre**
1. **Marco conceptual** desde la taxonomía de Baltrušaitis y la lectura estructural de Li & Tang.  
2. Baselines **solo-texto** e **solo-imagen**.  
3. **Fusión temprana por concatenación de características (features)**.  
4. **Fusión tardía** por combinación de probabilidades.  
5. **Fusión intermedia con compuertas**, inspirada en GMU.  
6. Extensiones conceptuales hacia **FiLM** y **Tensor Fusion**.  
7. Evaluación, análisis de error, robustez a modalidad ausente y discusión metodológica.

#### **Hilo conductor**
Una buena manera de leer la semana es como una evolución de cuatro ideas de fusión intermedia:

**concat + MLP -> GMU -> FiLM -> TFN**

- **concat + MLP**: junta modalidades,  
- **GMU**: junta modalidades y aprende cuánto pesa cada una,  
- **FiLM**: una modalidad condiciona las features internas de otra, 
- **TFN**: explicita interacciones unimodales, bimodales y trimodales.

#### **Qué implementa el cuaderno**
Este laboratorio implementa directamente:
- solo-texto,
- solo-imagen,
- fusión temprana por concatenación,
- fusión tardía,
- una variante de **gating**.

Además incorpora, vía celdas teóricas, el encuadre de:
- **GMU** ([Arevalo](https://github.com/kapumota/MCC225/blob/main/Semana3/papers/Arevalo.pdf)),
- **FiLM** ([Perez](https://github.com/kapumota/MCC225/blob/main/Semana3/papers/Perez.pdf)),
- **TFN** ([Zadeh](https://github.com/kapumota/MCC225/blob/main/Semana3/papers/Zadeh.pdf)).

#### **Resultado esperado**
Al final del cuaderno deberías poder responder, con evidencia experimental y conceptual:

> ¿cuándo la multimodalidad realmente añade complementariedad y cuándo solo replica la modalidad dominante?

### **1. Marco conceptual**

#### **1.1. De representación a fusión**

Una idea central del survey de **Baltrušaitis, Ahuja y Morency** es que antes de hablar de **fusión** (*fusion*) conviene fijar cómo viven las modalidades en el espacio de representación. En particular, distinguen entre:

* **joint representations** (*representaciones conjuntas*): varias modalidades se proyectan a una representación compartida.
* **coordinated representations** (*representaciones coordinadas*): cada modalidad conserva su propio espacio, pero estos espacios se coordinan mediante restricciones de similitud, correlación o estructura.

Ese punto importa porque muchas estrategias de **fusión multimodal** (*multimodal fusion*) son, en el fondo, maneras diferentes de construir una representación conjunta.

#### **1.2. La clasificación clásica y la estructural**

En la literatura clásica suele hablarse de:

* **early fusion** (*fusión temprana*),
* **late fusion** (*fusión tardía*),
* y variantes híbridas (*hybrid fusion*).

En cambio, **Li & Tang** proponen una lectura más arquitectónica:

* **data-level fusion** (*fusión a nivel de datos*),
* **feature-level fusion** (*fusión a nivel de características*),
* **output-level fusion** (*fusión a nivel de salida*).

En este cuaderno trabajaremos sobre todo con:

* **feature-level fusion** (*fusión a nivel de características*): los embeddings unimodales se integran después de ser extraídos por modelos específicos para cada modalidad.
* **output-level fusion** (*fusión a nivel de salida*): se combinan de forma tardía las probabilidades, predicciones o puntajes producidos por modelos unimodales.

#### **1.3. Evolución de ideas de fusión intermedia**

Las lecturas avanzadas de la semana permiten leer la **fusión intermedia** (*intermediate fusion*) como una progresión:

1. **concat + MLP** (*concatenación + perceptrón multicapa*)
   Cada modalidad produce un **embedding** (*representación vectorial*) y luego los embeddings se concatenan.

2. **GMU-Gated Multimodal Unit** (*unidad multimodal con compuerta*)
   Se aprende una **gate** (*compuerta*) que decide cuánto confiar en cada modalidad para cada ejemplo.

3. **FiLM- Feature-wise Linear Modulation** (*modulación lineal por característica*)
   Una modalidad condiciona las **features** (*características internas*) de otra mediante una transformación afín aplicada característica por característica:

   $$
   F' = \gamma F + \beta
   $$

   En este caso, los parámetros $\gamma$ y $\beta$ permiten modular la representación de una modalidad usando información de otra.

4. **TFN-Tensor Fusion Network** (*red de fusión tensorial*)
   La representación multimodal contiene explícitamente interacciones:

   * **unimodal interactions** (*interacciones unimodales*),
   * **bimodal interactions** (*interacciones bimodales*),
   * **trimodal interactions** (*interacciones trimodales*).

#### **1.4. Qué implementa este laboratorio y qué no**

Este cuaderno **sí implementa**:

* **unimodal baselines** (*modelos base unimodales*),
* **early fusion by concatenation** (*fusión temprana por concatenación*),
* **late fusion** (*fusión tardía*),
* una variante clara y didáctica de **gated fusion** (*fusión con compuertas*).

Este cuaderno **no implementa de forma exacta**:

* **GMU** (*Gated Multimodal Unit*/*unidad multimodal con compuerta*) original,
* **FiLM** (*Feature-wise Linear Modulation*/*modulación lineal por característica*),
* **Tensor Fusion** (*fusión tensorial*).

Sin embargo, sí deja preparado el marco conceptual para entenderlos y compararlos.

#### **1.5. Pregunta guía**

La semana se organiza alrededor de esta pregunta:

> ¿Qué ganamos y qué perdemos cuando fusionamos modalidades, y qué tipo de interacción entre ellas estamos modelando realmente?


#### **2. Formato de datos asumido**

Este cuaderno asume un problema de **clasificación supervisada** con una tabla por partición:

- `train.csv`
- `val.csv`
- `test.csv`

Cada archivo debe contener al menos estas columnas:

- `text`: texto asociado a la instancia  
- `image_path`: ruta local a la imagen  
- `label`: clase objetivo  

##### **Estructura sugerida**
```text
data/
├── train.csv
├── val.csv
├── test.csv
└── images/
    ├── img_0001.jpg
    ├── img_0002.jpg
    └── ...
```

**Observación**
Si deseas trabajar con **MM-IMDb** simplificado, basta con adaptar los CSV para que cada fila tenga:
- el **plot** o sinopsis en `text`,
- la ruta al **poster** en `image_path`,
- la etiqueta en `label`.

**Recomendación**
Para la primera implementación, conviene usar una versión **multiclase reducida** con pocas clases bien representadas, antes de escalar a multi-label.

> Nota: este cuaderno corregido intenta resolver automáticamente rutas relativas inconsistentes como `images/...` vs `data/images/...`, y reporta/elimina filas con imágenes inexistentes antes de construir el `DataLoader`.

In [ ]:
# 3. Importaciones
from __future__ import annotations

import os
import json
import math
import random
import warnings
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image
from tqdm.auto import tqdm

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, StandardScaler

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import torchvision
from torchvision import transforms
from torchvision.models import resnet18, ResNet18_Weights

warnings.filterwarnings("ignore")

In [ ]:
# 4. Configuración general
@dataclass
class Config:
    data_dir: str = "data"
    train_file: str = "train.csv"
    val_file: str = "val.csv"
    test_file: str = "test.csv"

    text_col: str = "text"
    image_col: str = "image_path"
    label_col: str = "label"

    random_seed: int = 42
    batch_size: int = 32
    # 0 facilita la depuración de rutas y evita errores envueltos por workers.
    num_workers: int = 0

    text_max_features: int = 20000
    text_svd_dim: int = 256

    image_size: int = 224
    image_embed_dim: int = 512

    clf_max_iter: int = 2000
    gated_hidden_dim: int = 256
    gated_epochs: int = 15
    gated_lr: float = 1e-3
    gated_weight_decay: float = 1e-4

CFG = Config()
CFG

In [ ]:
# 5. Reproducibilidad
def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(CFG.random_seed)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Dispositivo:", DEVICE)

In [ ]:
# 6. Carga y validación de datos
def resolve_image_path(path_value: str | Path) -> str:
    raw = Path(str(path_value))

    candidates = []
    if raw.is_absolute():
        candidates.append(raw)
    else:
        candidates.extend([
            raw,                               # images/xxx.png
            Path(CFG.data_dir) / raw,          # data/images/xxx.png
            Path(CFG.data_dir) / "images" / raw.name,  # data/images/xxx.png si el CSV trae solo el nombre o una ruta no alineada
            Path("images") / raw.name,         # images/xxx.png usando solo el nombre
        ])

    seen = set()
    unique_candidates = []
    for cand in candidates:
        cand = Path(cand)
        key = str(cand)
        if key not in seen:
            unique_candidates.append(cand)
            seen.add(key)

    for cand in unique_candidates:
        if cand.exists():
            return str(cand)

    # Si no existe todavía, devolvemos la opción más probable para facilitar la depuración.
    if raw.is_absolute():
        return str(raw)
    return str(Path(CFG.data_dir) / raw)


def load_split(csv_path: Path) -> pd.DataFrame:
    if not csv_path.exists():
        raise FileNotFoundError(f"No se encontró el archivo: {csv_path}")

    df = pd.read_csv(csv_path)
    required_cols = {CFG.text_col, CFG.image_col, CFG.label_col}
    missing = required_cols - set(df.columns)
    if missing:
        raise ValueError(f"Faltan columnas requeridas en {csv_path.name}: {missing}")

    df = df.copy()
    df[CFG.text_col] = df[CFG.text_col].fillna("").astype(str)
    df[CFG.image_col] = df[CFG.image_col].astype(str).apply(resolve_image_path)
    df[CFG.label_col] = df[CFG.label_col].astype(str)
    return df


def report_and_clean_missing_images(df: pd.DataFrame, split_name: str) -> pd.DataFrame:
    exists_mask = df[CFG.image_col].apply(lambda p: Path(p).exists())
    missing = int((~exists_mask).sum())
    print(f"Imágenes faltantes en {split_name}: {missing}")

    if missing > 0:
        display(df.loc[~exists_mask, [CFG.image_col, CFG.label_col]].head(10))
        df = df.loc[exists_mask].reset_index(drop=True)
        print(f"{split_name}: se eliminaron {missing} filas con imágenes inexistentes.")

    return df


data_dir = Path(CFG.data_dir)
train_df = load_split(data_dir / CFG.train_file)
val_df = load_split(data_dir / CFG.val_file)
test_df = load_split(data_dir / CFG.test_file)

train_df = report_and_clean_missing_images(train_df, "train")
val_df = report_and_clean_missing_images(val_df, "val")
test_df = report_and_clean_missing_images(test_df, "test")

print("Directorio de trabajo actual:", Path.cwd())
print("Train:", train_df.shape)
print("Val  :", val_df.shape)
print("Test :", test_df.shape)
train_df.head()

In [ ]:
# 7. Inspección básica del dataset
def summarize_split(name: str, df: pd.DataFrame) -> None:
    print(f"\n{name}")
    print("-" * len(name))
    print("Número de muestras:", len(df))
    print("Clases:", df[CFG.label_col].nunique())
    print(df[CFG.label_col].value_counts().head(10))

summarize_split("Train", train_df)
summarize_split("Val", val_df)
summarize_split("Test", test_df)

In [ ]:
# 8. Visualización rápida de ejemplos
def show_examples(df: pd.DataFrame, n: int = 3) -> None:
    sampled = df.sample(min(n, len(df)), random_state=CFG.random_seed).reset_index(drop=True)
    fig, axes = plt.subplots(len(sampled), 2, figsize=(12, 4 * len(sampled)))
    if len(sampled) == 1:
        axes = np.array([axes])

    for i, row in sampled.iterrows():
        image_path = Path(row[CFG.image_col])
        axes[i, 0].axis("off")
        if image_path.exists():
            img = Image.open(image_path).convert("RGB")
            axes[i, 0].imshow(img)
            axes[i, 0].set_title(f"Imagen · {row[CFG.label_col]}")
        else:
            axes[i, 0].text(0.5, 0.5, f"No existe\n{image_path}", ha="center", va="center")
            axes[i, 0].set_title("Imagen no encontrada")

        axes[i, 1].axis("off")
        txt = row[CFG.text_col][:1200]
        axes[i, 1].text(0.0, 1.0, txt, ha="left", va="top", wrap=True)
        axes[i, 1].set_title("Texto")

    plt.tight_layout()

show_examples(train_df, n=2)

#### **3. Baseline 1 - clasificación con texto**

**Idea**
Partimos de un baseline fuerte, simple y reproducible:
- representación: **TF-IDF**
- clasificador: **Logistic Regression**

**Justificación**
Para una primera línea base, el objetivo no es maximizar SOTA, sino:
- obtener una referencia interpretable,
- establecer una comparación limpia con solo-imagen,
- medir cuánta señal semántica aporta el lenguaje por sí solo antes de cualquier interacción multimodal.

**Lectura conceptual**

Este baseline aísla la **dinámica intra-modal** de la modalidad lingüística: qué tanto puede hacer el texto sin ayuda de la imagen.

In [ ]:
# 9. Etiquetas y utilidades de evaluación
label_encoder = LabelEncoder()
y_train = label_encoder.fit_transform(train_df[CFG.label_col])
y_val = label_encoder.transform(val_df[CFG.label_col])
y_test = label_encoder.transform(test_df[CFG.label_col])

class_names = list(label_encoder.classes_)
num_classes = len(class_names)
print("Clases codificadas:", class_names)

def evaluate_predictions(y_true: np.ndarray, y_pred: np.ndarray, split_name: str) -> Dict[str, float]:
    metrics = {
        "split": split_name,
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro"),
        "weighted_f1": f1_score(y_true, y_pred, average="weighted"),
    }
    return metrics

def print_report(y_true: np.ndarray, y_pred: np.ndarray, title: str) -> None:
    print(f"\n{title}")
    print(classification_report(y_true, y_pred, target_names=class_names, digits=4))

def plot_conf_matrix(y_true: np.ndarray, y_pred: np.ndarray, title: str) -> None:
    cm = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(7, 6))
    im = ax.imshow(cm)
    ax.set_title(title)
    ax.set_xlabel("Predicción")
    ax.set_ylabel("Real")
    ax.set_xticks(range(len(class_names)))
    ax.set_yticks(range(len(class_names)))
    ax.set_xticklabels(class_names, rotation=45, ha="right")
    ax.set_yticklabels(class_names)
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, cm[i, j], ha="center", va="center")
    plt.colorbar(im, ax=ax)
    plt.tight_layout()

In [ ]:
# 10. solo-texto
text_clf = Pipeline(
    steps=[
        ("tfidf", TfidfVectorizer(
            max_features=CFG.text_max_features,
            ngram_range=(1, 2),
            min_df=2,
            sublinear_tf=True,
        )),
        ("clf", LogisticRegression(
            max_iter=CFG.clf_max_iter,
            class_weight="balanced",
            n_jobs=None,
        )),
    ]
)

text_clf.fit(train_df[CFG.text_col], y_train)

val_pred_text = text_clf.predict(val_df[CFG.text_col])
test_pred_text = text_clf.predict(test_df[CFG.text_col])

text_val_metrics = evaluate_predictions(y_val, val_pred_text, "val_text")
text_test_metrics = evaluate_predictions(y_test, test_pred_text, "test_text")

print(text_val_metrics)
print(text_test_metrics)
print_report(y_test, test_pred_text, "Reporte: solo-texto (test)")
plot_conf_matrix(y_test, test_pred_text, "Matriz de confusión: Solo-texto")

#### **4. Baseline 2 - clasificación con imagen**

**Idea**

Usamos una CNN preentrenada como extractor de características:
- **ResNet18** preentrenada,
- congelada,
- usando el embedding del penúltimo nivel,
- con un clasificador lineal encima.

**Justificación**

Esto separa claramente:
1. representación visual,
2. decisión de clasificación.

Ese desacople ayuda mucho cuando luego fusionamos, porque permite identificar si la mejora proviene de la modalidad visual en sí o de la interacción multimodal.

#### **Lectura conceptual**

Este baseline aísla la **dinámica intra-modal** de la visión: cuánto puede resolver la imagen por sí sola.

In [ ]:
# 11. Dataset de imágenes y extractor de embeddings
class ImageDataset(Dataset):
    def __init__(self, df: pd.DataFrame, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self) -> int:
        return len(self.df)

    def __getitem__(self, idx: int):
        row = self.df.iloc[idx]
        image_path = Path(row[CFG.image_col])

        if not image_path.exists():
            raise FileNotFoundError(
                f"No se encontró la imagen: {image_path}\n"
                f"Directorio de trabajo actual: {Path.cwd()}"
            )

        image = Image.open(image_path).convert("RGB")
        if self.transform is not None:
            image = self.transform(image)
        label = row[CFG.label_col]
        return image, label, idx

try:
    weights = ResNet18_Weights.DEFAULT
    image_transform = weights.transforms()
    backbone = resnet18(weights=weights)
except Exception as e:
    print("No fue posible cargar pesos preentrenados, se usarán transformaciones estándar.")
    print("Detalle:", e)
    image_transform = transforms.Compose([
        transforms.Resize((CFG.image_size, CFG.image_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225]),
    ])
    backbone = resnet18(weights=None)

backbone.fc = nn.Identity()
backbone = backbone.to(DEVICE)
backbone.eval()

In [ ]:
# 12. Función para extraer embeddings de imagen
@torch.no_grad()
def extract_image_embeddings(df: pd.DataFrame) -> np.ndarray:
    if len(df) == 0:
        raise ValueError("El DataFrame no contiene filas válidas para extraer embeddings de imagen.")

    ds = ImageDataset(df, transform=image_transform)
    loader = DataLoader(
        ds,
        batch_size=CFG.batch_size,
        shuffle=False,
        num_workers=CFG.num_workers,
        pin_memory=torch.cuda.is_available(),
    )
    all_embeddings = []
    for images, _, _ in tqdm(loader, desc="Extrayendo embeddings de imagen"):
        images = images.to(DEVICE)
        emb = backbone(images)
        all_embeddings.append(emb.cpu().numpy())
    return np.concatenate(all_embeddings, axis=0)

X_train_img = extract_image_embeddings(train_df)
X_val_img = extract_image_embeddings(val_df)
X_test_img = extract_image_embeddings(test_df)

print("Embeddings imagen train:", X_train_img.shape)

In [ ]:
# 13. solo-imagen
image_clf = Pipeline(
    steps=[
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(
            max_iter=CFG.clf_max_iter,
            class_weight="balanced",
        )),
    ]
)

image_clf.fit(X_train_img, y_train)

val_pred_img = image_clf.predict(X_val_img)
test_pred_img = image_clf.predict(X_test_img)

image_val_metrics = evaluate_predictions(y_val, val_pred_img, "val_image")
image_test_metrics = evaluate_predictions(y_test, test_pred_img, "test_image")

print(image_val_metrics)
print(image_test_metrics)
print_report(y_test, test_pred_img, "Reporte: Solo-imagen (test)")
plot_conf_matrix(y_test, test_pred_img, "Matriz de confusión: Solo-imagen")

#### **5. Fusión temprana por concatenación de características**

**Idea**

Concatenamos una representación textual densa con una representación visual densa.

**Diseño**
1. Extraer **TF-IDF**.  
2. Reducir dimensión con **TruncatedSVD**.  
3. Concatenar con embeddings de imagen.  
4. Entrenar un clasificador sobre el vector fusionado.

**Comentario**
La reducción dimensional del texto evita que la modalidad textual domine por escala y dimensionalidad.

**Lectura conceptual**

Esta es la forma más simple de **feature-level fusion**.  
Preserva directamente la información unimodal, pero deja las interacciones entre modalidades como algo **implícito** que el clasificador posterior debe descubrir.

In [ ]:
# 14. Representación textual densa para fusión
tfidf = TfidfVectorizer(
    max_features=CFG.text_max_features,
    ngram_range=(1, 2),
    min_df=2,
    sublinear_tf=True,
)

X_train_text_sparse = tfidf.fit_transform(train_df[CFG.text_col])
X_val_text_sparse = tfidf.transform(val_df[CFG.text_col])
X_test_text_sparse = tfidf.transform(test_df[CFG.text_col])

svd_dim = min(CFG.text_svd_dim, X_train_text_sparse.shape[1] - 1) if X_train_text_sparse.shape[1] > 1 else 1
text_svd = TruncatedSVD(n_components=svd_dim, random_state=CFG.random_seed)

X_train_text_dense = text_svd.fit_transform(X_train_text_sparse)
X_val_text_dense = text_svd.transform(X_val_text_sparse)
X_test_text_dense = text_svd.transform(X_test_text_sparse)

print("Texto denso train:", X_train_text_dense.shape)
print("Varianza explicada acumulada SVD:", text_svd.explained_variance_ratio_.sum())

In [ ]:
# 15. Early fusion: concatenación

X_train_early = np.concatenate([X_train_text_dense, X_train_img], axis=1)
X_val_early = np.concatenate([X_val_text_dense, X_val_img], axis=1)
X_test_early = np.concatenate([X_test_text_dense, X_test_img], axis=1)

early_clf = Pipeline(
    steps=[
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(
            max_iter=CFG.clf_max_iter,
            class_weight="balanced",
        )),
    ]
)

early_clf.fit(X_train_early, y_train)

val_pred_early = early_clf.predict(X_val_early)
test_pred_early = early_clf.predict(X_test_early)

early_val_metrics = evaluate_predictions(y_val, val_pred_early, "val_early")
early_test_metrics = evaluate_predictions(y_test, test_pred_early, "test_early")

print(early_val_metrics)
print(early_test_metrics)
print_report(y_test, test_pred_early, "Reporte: Early Fusion (test)")
plot_conf_matrix(y_test, test_pred_early, "Matriz de confusion: Early Fusion")

#### **6. Fusión tardía**

**Idea**

Entrenamos modelos unimodales por separado y combinamos sus probabilidades:
- promedio simple,
- o promedio ponderado.

**Ventaja**

Es una fusión modular, fácil de explicar y de depurar.

**Limitación**

No modela interacciones profundas entre modalidades, porque combina decisiones ya formadas.

**Lectura conceptual**

En la clasificación estructural de [Li & Tang](https://github.com/kapumota/MCC225/blob/main/Semana3/papers/Li.pdf), esto cae del lado de **output-level fusion**.

In [ ]:
# 16. Probabilidades unimodales
val_proba_text = text_clf.predict_proba(val_df[CFG.text_col])
test_proba_text = text_clf.predict_proba(test_df[CFG.text_col])

val_proba_img = image_clf.predict_proba(X_val_img)
test_proba_img = image_clf.predict_proba(X_test_img)

def late_fusion_predict(proba_a: np.ndarray, proba_b: np.ndarray, alpha: float = 0.5) -> np.ndarray:
    fused = alpha * proba_a + (1 - alpha) * proba_b
    return fused.argmax(axis=1)

candidate_alphas = np.linspace(0.0, 1.0, 11)
best_alpha, best_macro_f1 = None, -1.0

for alpha in candidate_alphas:
    pred = late_fusion_predict(val_proba_text, val_proba_img, alpha=alpha)
    score = f1_score(y_val, pred, average="macro")
    if score > best_macro_f1:
        best_macro_f1 = score
        best_alpha = float(alpha)

print("Mejor alpha en validación:", best_alpha)
print("Macro-F1 validación:", best_macro_f1)

In [ ]:
# 17. Evaluación de late fusion
val_pred_late = late_fusion_predict(val_proba_text, val_proba_img, alpha=best_alpha)
test_pred_late = late_fusion_predict(test_proba_text, test_proba_img, alpha=best_alpha)

late_val_metrics = evaluate_predictions(y_val, val_pred_late, "val_late")
late_test_metrics = evaluate_predictions(y_test, test_pred_late, "test_late")

print(late_val_metrics)
print(late_test_metrics)
print_report(y_test, test_pred_late, "Reporte:Late Fusion (test)")
plot_conf_matrix(y_test, test_pred_late, "Matriz de confusión: Late Fusion")

#### **7. Fusión intermedia con compuertas: aproximación a GMU**

**Motivación**

En muchos problemas, una modalidad no aporta siempre lo mismo.  
La idea del **gating** es aprender cuánto confiar en cada modalidad para cada ejemplo.

##### **Diseño de esta implementación**

- proyectamos texto e imagen a una dimensión común,
- aprendemos una compuerta `g` entre 0 y 1,
- combinamos ambas proyecciones como:

$$
h = g \odot h_{text} + (1-g) \odot h_{img}
$$

- sobre `h` entrenamos un clasificador final.

##### **Relación con GMU**

Esta implementación **no pretende replicar exactamente** el GMU (Gated Multimodal Unit) de [Arevalo](https://github.com/kapumota/MCC225/blob/main/Semana3/papers/Arevalo.pdf), pero preserva su idea central:

> la mezcla entre modalidades no debe ser fija, sino depender de la entrada.

**Lectura conceptual**

- **concat + MLP**: junta modalidades, 
- **gating**: junta modalidades y además decide cuánto pesa cada una.

In [ ]:
# 18. Dataset tabular para gating
class MultimodalFeatureDataset(Dataset):
    def __init__(self, text_features: np.ndarray, image_features: np.ndarray, labels: np.ndarray):
        self.text_features = torch.tensor(text_features, dtype=torch.float32)
        self.image_features = torch.tensor(image_features, dtype=torch.float32)
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx: int):
        return self.text_features[idx], self.image_features[idx], self.labels[idx]


class GatedFusionClassifier(nn.Module):
    def __init__(self, text_dim: int, image_dim: int, hidden_dim: int, num_classes: int):
        super().__init__()
        self.text_proj = nn.Sequential(
            nn.Linear(text_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.2),
        )
        self.image_proj = nn.Sequential(
            nn.Linear(image_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.2),
        )
        # Gate sobre features ya proyectadas: más cercano al espíritu de GMU
        self.gate = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.Sigmoid(),
        )
        self.classifier = nn.Linear(hidden_dim, num_classes)

    def forward(self, text_x: torch.Tensor, image_x: torch.Tensor, return_gate: bool = False):
        t = self.text_proj(text_x)
        i = self.image_proj(image_x)
        g = self.gate(torch.cat([t, i], dim=1))
        fused = g * t + (1.0 - g) * i
        logits = self.classifier(fused)
        if return_gate:
            return logits, g
        return logits

In [ ]:
# 19. Entrenamiento del modelo con gating
train_ds = MultimodalFeatureDataset(X_train_text_dense, X_train_img, y_train)
val_ds = MultimodalFeatureDataset(X_val_text_dense, X_val_img, y_val)
test_ds = MultimodalFeatureDataset(X_test_text_dense, X_test_img, y_test)

train_loader = DataLoader(train_ds, batch_size=CFG.batch_size, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=CFG.batch_size, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=CFG.batch_size, shuffle=False)

gated_model = GatedFusionClassifier(
    text_dim=X_train_text_dense.shape[1],
    image_dim=X_train_img.shape[1],
    hidden_dim=CFG.gated_hidden_dim,
    num_classes=num_classes,
).to(DEVICE)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(
    gated_model.parameters(),
    lr=CFG.gated_lr,
    weight_decay=CFG.gated_weight_decay,
)

def run_epoch(model, loader, optimizer=None):
    is_train = optimizer is not None
    model.train(is_train)

    total_loss = 0.0
    all_preds, all_labels = [], []

    for text_x, image_x, labels in loader:
        text_x = text_x.to(DEVICE)
        image_x = image_x.to(DEVICE)
        labels = labels.to(DEVICE)

        logits = model(text_x, image_x)
        loss = criterion(logits, labels)

        if is_train:
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        total_loss += loss.item() * labels.size(0)
        preds = logits.argmax(dim=1)
        all_preds.append(preds.detach().cpu().numpy())
        all_labels.append(labels.detach().cpu().numpy())

    all_preds = np.concatenate(all_preds)
    all_labels = np.concatenate(all_labels)

    metrics = {
        "loss": total_loss / len(loader.dataset),
        "accuracy": accuracy_score(all_labels, all_preds),
        "macro_f1": f1_score(all_labels, all_preds, average="macro"),
        "weighted_f1": f1_score(all_labels, all_preds, average="weighted"),
    }
    return metrics, all_preds, all_labels

best_state = None
best_val_f1 = -1.0
history = []

for epoch in range(1, CFG.gated_epochs + 1):
    train_metrics, _, _ = run_epoch(gated_model, train_loader, optimizer=optimizer)
    val_metrics, _, _ = run_epoch(gated_model, val_loader, optimizer=None)
    history.append({
        "epoch": epoch,
        "train_loss": train_metrics["loss"],
        "train_macro_f1": train_metrics["macro_f1"],
        "val_loss": val_metrics["loss"],
        "val_macro_f1": val_metrics["macro_f1"],
    })
    print(
        f"Epoch {epoch:02d} | "
        f"train_loss={train_metrics['loss']:.4f} | "
        f"train_macro_f1={train_metrics['macro_f1']:.4f} | "
        f"val_loss={val_metrics['loss']:.4f} | "
        f"val_macro_f1={val_metrics['macro_f1']:.4f}"
    )
    if val_metrics["macro_f1"] > best_val_f1:
        best_val_f1 = val_metrics["macro_f1"]
        best_state = {k: v.cpu().clone() for k, v in gated_model.state_dict().items()}

if best_state is not None:
    gated_model.load_state_dict(best_state)

In [ ]:
# 20. Curva de entrenamiento y evaluación final de gating
history_df = pd.DataFrame(history)
display(history_df.tail())

plt.figure(figsize=(8, 4))
plt.plot(history_df["epoch"], history_df["train_macro_f1"], label="train_macro_f1")
plt.plot(history_df["epoch"], history_df["val_macro_f1"], label="val_macro_f1")
plt.xlabel("Epoch")
plt.ylabel("Macro-F1")
plt.title("Entrenamiento · Gated Fusion")
plt.legend()
plt.show()

val_metrics_gated, val_pred_gated, _ = run_epoch(gated_model, val_loader, optimizer=None)
test_metrics_gated, test_pred_gated, test_labels_gated = run_epoch(gated_model, test_loader, optimizer=None)

gated_val_metrics = {"split": "val_gated", **val_metrics_gated}
gated_test_metrics = {"split": "test_gated", **test_metrics_gated}

print(gated_val_metrics)
print(gated_test_metrics)
print_report(test_labels_gated, test_pred_gated, "Reporte :Gated Fusion (test)")
plot_conf_matrix(test_labels_gated, test_pred_gated, "Matriz de confusión: Gated Fusion")

#### **8. Inspección del gate aprendido**

Una ventaja de la fusión con compuertas es que podemos inspeccionar **cómo reparte confianza** entre texto e imagen.

En esta implementación:
- valores altos del gate favorecen la rama textual proyectada,
- valores bajos favorecen la rama visual proyectada.

Esto no equivale a una explicación causal completa, pero sí da una señal útil sobre qué modalidad está dominando la representación intermedia.

In [ ]:
# 20. Inspección del gate aprendido
@torch.no_grad()
def collect_gate_statistics(model, loader):
    model.eval()
    all_gates = []
    all_labels = []
    for text_x, image_x, labels in loader:
        text_x = text_x.to(DEVICE)
        image_x = image_x.to(DEVICE)
        logits, gates = model(text_x, image_x, return_gate=True)
        all_gates.append(gates.cpu().numpy())
        all_labels.append(labels.numpy())
    gates = np.concatenate(all_gates, axis=0)
    labels = np.concatenate(all_labels, axis=0)
    gate_mean = gates.mean(axis=1)
    return gates, gate_mean, labels

gates_val, gate_mean_val, gate_labels_val = collect_gate_statistics(gated_model, val_loader)

plt.figure(figsize=(8, 4))
plt.hist(gate_mean_val, bins=30, edgecolor="black")
plt.title("Distribución del gate medio por ejemplo (validación)")
plt.xlabel("Gate medio")
plt.ylabel("Frecuencia")
plt.show()

gate_df = pd.DataFrame({
    "label": label_encoder.inverse_transform(gate_labels_val),
    "gate_mean": gate_mean_val,
})
gate_by_class = gate_df.groupby("label")["gate_mean"].agg(["mean", "std", "count"]).sort_values("mean", ascending=False)
gate_by_class

#### **9. Más allá del gating: FiLM como condicionante cruzado**

Si el gating decide **cuánto pesa cada modalidad**, **FiLM** propone una idea más fina:

> una modalidad puede **condicionar internamente** el procesamiento de otra.

En [Perez](https://github.com/kapumota/MCC225/blob/main/Semana3/papers/Perez.pdf), el lenguaje genera parámetros $\gamma$ y $\beta$ que modulan **feature map (mapa de características)** visuales:

$$
F' = \gamma F + \beta
$$

La diferencia clave con el gating es que aquí no solo se mezcla texto e imagen al final, sino que una modalidad **reconfigura capa por capa** la computación de la otra.

#### **10. Más allá de concatenar: Tensor Fusion**

**Tensor Fusion Network (TFN)** propone otra idea:

> representar explícitamente las interacciones entre modalidades, en vez de dejarlas implícitas dentro de una concatenación.

La concatenación clásica usa solo:

$$
[z^l, z^v, z^a]
$$

En cambio, Tensor Fusion construye una representación con:
- términos **unimodales**,
- términos **bimodales**,
- y un término **trimodal**.

De ese modo, la representación conserva explícitamente:
- $z^l$,
- $z^v$,
- $z^a$,
- $z^l \otimes z^v$,
- $z^l \otimes z^a$,
- $z^v \otimes z^a$,
- $z^l \otimes z^v \otimes z^a$.

En esta lectura, la concatenación aparece como un caso especial que solo preserva las partes unimodales.

##### **10.1. Resumen de la evolución: concat + MLP -> GMU -> FiLM -> TFN**

- **concat + MLP**: junta modalidades, las interacciones quedan implícitas.  
- **GMU**: junta modalidades y aprende cuánto pesa cada una para cada ejemplo.  
- **FiLM**: una modalidad modula las features internas de otra.  
- **TFN**: la representación contiene explícitamente interacciones de orden 1, 2 y 3.

Esta progresión resume cuatro maneras cada vez más expresivas de hacer **fusión intermedia**.

#### **11. Comparación consolidada de modelos**

En este punto ya podemos comparar cinco escenarios implementados:
1. solo-texto,
2. solo-imagen,
3. fusión temprana por concatenación,
4. fusión tardía,
5. gated.

La pregunta importante no es solo cuál gana, sino:
- cuánto gana,
- frente a qué baseline,
- y qué tipo de interacción entre modalidades está modelando.

In [ ]:
# 21. Tabla comparativa de resultados
results_df = pd.DataFrame([
    text_test_metrics,
    image_test_metrics,
    early_test_metrics,
    late_test_metrics,
    gated_test_metrics,
])

results_df = results_df[["split", "accuracy", "macro_f1", "weighted_f1"]].sort_values(
    by="macro_f1", ascending=False
).reset_index(drop=True)

results_df

##### **11.1. Robustez a modalidad ausente o corrupta**

Una forma rápida de evaluar si la fusión aprendió verdadera complementariedad es perturbar una modalidad en el conjunto de prueba:

- **texto ausente**: reemplazamos las features textuales por cero,
- **imagen ausente**: reemplazamos las features visuales por cero,
- **ambas presentes**: condición normal.

Esto no sustituye una evaluación sistemática de *missing modality*, pero sirve como prueba de estrés.

In [ ]:
# 22. Robustez simple a modalidad ausente en gated fusion
@torch.no_grad()
def evaluate_gated_with_mask(model, text_features, image_features, labels, drop_text=False, drop_image=False):
    model.eval()
    text_x = torch.tensor(text_features, dtype=torch.float32).to(DEVICE)
    image_x = torch.tensor(image_features, dtype=torch.float32).to(DEVICE)
    if drop_text:
        text_x = torch.zeros_like(text_x)
    if drop_image:
        image_x = torch.zeros_like(image_x)
    logits = model(text_x, image_x)
    preds = logits.argmax(dim=1).cpu().numpy()
    return {
        "accuracy": accuracy_score(labels, preds),
        "macro_f1": f1_score(labels, preds, average="macro"),
        "weighted_f1": f1_score(labels, preds, average="weighted"),
    }

stress_results = pd.DataFrame([
    {"condicion": "normal", **evaluate_gated_with_mask(gated_model, X_test_text_dense, X_test_img, y_test)},
    {"condicion": "sin_texto", **evaluate_gated_with_mask(gated_model, X_test_text_dense, X_test_img, y_test, drop_text=True)},
    {"condicion": "sin_imagen", **evaluate_gated_with_mask(gated_model, X_test_text_dense, X_test_img, y_test, drop_image=True)},
])

stress_results

#### **12. Análisis de error**

Una línea base multimodal seria no termina en la tabla de métricas.  
También debe responder preguntas como:

1. ¿Qué clases se benefician de la fusión?  
2. ¿Hay casos donde el texto es suficiente y la imagen añade ruido?  
3. ¿Hay casos donde la imagen corrige ambigüedades del texto?  
4. ¿La fusión temprana supera consistentemente a la tardía?  
5. ¿El gating ayuda cuando una modalidad es poco confiable?.

**Tipología útil de errores**

Conviene distinguir al menos tres tipos de fallo:
- **error de representación**: una modalidad aislada ya no captura la señal,
- **error de fusión**: las modalidades contienen información útil, pero el mecanismo de integración no la combina bien,
- **error de alineamiento semántico**: texto e imagen parecen hablar de cosas distintas o una modalidad añade ruido.

In [ ]:
# 23. Tabla de predicciones para inspección de errores
test_analysis = test_df.copy().reset_index(drop=True)
test_analysis["y_true"] = label_encoder.inverse_transform(y_test)
test_analysis["pred_text"] = label_encoder.inverse_transform(test_pred_text)
test_analysis["pred_image"] = label_encoder.inverse_transform(test_pred_img)
test_analysis["pred_early"] = label_encoder.inverse_transform(test_pred_early)
test_analysis["pred_late"] = label_encoder.inverse_transform(test_pred_late)
test_analysis["pred_gated"] = label_encoder.inverse_transform(test_pred_gated)

test_analysis["ok_text"] = test_analysis["y_true"] == test_analysis["pred_text"]
test_analysis["ok_image"] = test_analysis["y_true"] == test_analysis["pred_image"]
test_analysis["ok_early"] = test_analysis["y_true"] == test_analysis["pred_early"]
test_analysis["ok_late"] = test_analysis["y_true"] == test_analysis["pred_late"]
test_analysis["ok_gated"] = test_analysis["y_true"] == test_analysis["pred_gated"]

test_analysis.head()

In [ ]:
# 24. Casos interesantes
# Casos donde texto falla, imagen falla, pero una variante multimodal acierta
interesting = test_analysis[
    (~test_analysis["ok_text"]) &
    (~test_analysis["ok_image"]) &
    (test_analysis["ok_early"] | test_analysis["ok_late"] | test_analysis["ok_gated"])
].copy()

print("Casos interesantes encontrados:", len(interesting))
interesting[[CFG.text_col, CFG.image_col, "y_true", "pred_text", "pred_image", "pred_early", "pred_late", "pred_gated"]].head(10)

In [ ]:
# 25. Visualización de errores seleccionados
def show_error_cases(df: pd.DataFrame, n: int = 4) -> None:
    subset = df.head(n).reset_index(drop=True)
    if len(subset) == 0:
        print("No hay casos para mostrar.")
        return

    fig, axes = plt.subplots(len(subset), 2, figsize=(12, 4 * len(subset)))
    if len(subset) == 1:
        axes = np.array([axes])

    for i, row in subset.iterrows():
        axes[i, 0].axis("off")
        image_path = Path(row[CFG.image_col])
        if image_path.exists():
            img = Image.open(image_path).convert("RGB")
            axes[i, 0].imshow(img)
        else:
            axes[i, 0].text(0.5, 0.5, f"No existe\n{image_path}", ha="center", va="center")

        axes[i, 0].set_title(
            f"Real: {row['y_true']}\n"
            f"T: {row['pred_text']} | I: {row['pred_image']}\n"
            f"E: {row['pred_early']} | L: {row['pred_late']} | G: {row['pred_gated']}"
        )

        axes[i, 1].axis("off")
        axes[i, 1].text(
            0.0, 1.0, row[CFG.text_col][:1200],
            ha="left", va="top", wrap=True
        )
        axes[i, 1].set_title("Texto")

    plt.tight_layout()

show_error_cases(interesting, n=3)

#### **13. Preguntas a desarrollar**

1. ¿La multimodalidad mejora de forma consistente sobre el mejor baseline unimodal?.
2. ¿Qué estrategia de fusión ofrece la mejor relación entre simplicidad, interpretabilidad y desempeño?. 
3. ¿La modalidad textual domina a la visual o viceversa? ¿Cómo lo sabes?.  
4. ¿Qué evidencia hay de complementariedad entre modalidades?.
5. ¿Qué errores sugieren problemas de representación y cuáles sugieren problemas de fusión?. 
6. ¿Cómo cambiaría el diseño si el problema fuera multi-label en lugar de multiclase?.
7. ¿Qué riesgos metodológicos aparecen si una modalidad está ausente, corrupta o desalineada?.


#### **14. Extensión conceptual a audio y video**

**Qué se mantiene**

Los mismos grandes problemas siguen presentes:
- representación,
- alineamiento,
- fusión,
- evaluación.

**Qué cambia**

Audio y video introducen dificultades adicionales:
- **estructura temporal**,
- **longitud variable**,
- **asincronía** entre modalidades,
- mayor costo computacional.

> Primero entendemos la fusión en un caso relativamente estable y reproducible
> (texto + imagen). Luego extendemos esas ideas a modalidades temporales.


#### **15. Ejercicios de ampliación**

Estos ejercicios buscan que no solo ejecutes modelos de fusión multimodal, sino que también compares estrategias, interpretes resultados y reflexiones sobre sus ventajas y limitaciones.

##### **A. Extensión mínima: MLP en fusión temprano**

En la parte principal del cuaderno, la estrategia de **fusión temprano** combina las representaciones de texto e imagen y luego entrena un clasificador lineal, como una regresión logística. En este ejercicio, se propone reemplazar ese clasificador por un **MLP pequeño** para estudiar si una capa no lineal mejora el desempeño.

##### **Tarea**

1. Toma el vector fusionado por concatenación:
   $$
   z = [z_{\text{text}} , z_{\text{img}}]
   $$
2. Reemplaza la regresión logística por un MLP pequeño, por ejemplo:

   * capa lineal de entrada a dimensión oculta,
   * activación ReLU,
   * dropout opcional,
   * capa de salida al número de clases.
3. Entrena este modelo con los mismos conjuntos `train`, `val` y `test`.
4. Compara contra la regresión logística ya implementada.

##### **Sugerencia de arquitectura**

* Entrada: dimensión total de la concatenación.
* Capa oculta: 128 o 256 unidades.
* Activación: ReLU.
* Dropout: entre 0.1 y 0.3.
* Salida: número de clases.

##### **Qué analizar**

* ¿Mejora la accuracy o el macro-F1 respecto a regresión logística?,
* ¿El MLP parece sobreajustar más?,
* ¿La mejora, si existe, justifica el aumento de complejidad?.

##### **Entrega esperada**

* Tabla comparativa entre regresión logística y MLP.
* Breve comentario sobre cuándo conviene un clasificador lineal y cuándo uno no lineal.

#### **B. Extensión intermedia: fusión tardía ponderada por clase**

En la versión base de **fusión tardía**, se combinan las probabilidades de texto e imagen con un único parámetro global $\alpha$:

$$
p = \alpha, p_{\text{text}} + (1-\alpha), p_{\text{img}}
$$

En este ejercicio, se propone usar un peso distinto para cada clase.

##### **Tarea**

1. En vez de usar un solo $\alpha$, define un vector:
   $$
   \boldsymbol{\alpha} = [\alpha_1, \alpha_2, \dots, \alpha_C]
   $$
   donde $C$ es el número de clases.
2. Para cada clase $c$, combina:
   $$
   p_c = \alpha_c, p_{\text{text},c} + (1-\alpha_c), p_{\text{img},c}
   $$
3. Normaliza las probabilidades finales para que vuelvan a sumar 1.
4. Busca los mejores valores de $\alpha_c$ usando el conjunto de validación.

##### **Sugerencias prácticas**

* Empieza con una búsqueda simple sobre valores discretos, por ejemplo:
  $$
  \alpha_c \in {0.0, 0.1, 0.2, \dots, 1.0}
  $$
* Como aproximación inicial, optimiza cada clase por separado.
* Luego compara con la versión de un único $\alpha$.

##### **Qué analizar**

* ¿Hay clases donde el texto domina claramente?.
* ¿Hay clases donde la imagen es más informativa?.
* ¿La fusión por clase mejora métricas globales o solo algunas clases específicas?.

##### **Entrega esperada**

* Vector final de pesos por clase.
* Comparación con late fusion global.
* Comentario sobre interpretabilidad: qué nos dicen esos pesos acerca del dataset.


#### **C. Extensión avanzada: comparar distintas funciones de fusión**

Hasta ahora, la estrategia más simple ha sido la **concatenación**. En este ejercicio se propone comparar varias formas de combinar representaciones multimodales.

#### **Métodos a implementar**

#### **1. Concatenación**

La forma más básica:
$$
z = [z_{\text{text}} , z_{\text{img}}]
$$

Sirve como línea base.

#### **2. Gating**

Aprende una compuerta que decida cuánto confiar en cada modalidad:
$$
g = \sigma(W[z_{\text{text}} , z_{\text{img}}] + b)
$$
$$
z = g \odot z_{\text{text}} + (1-g) \odot z_{\text{img}}
$$

Aquí $g$ puede ser escalar o vectorial.

#### **3. Producto elemento a elemento**

Combina ambas representaciones multiplicando coordenada a coordenada:
$$
z = z_{\text{text}} \odot z_{\text{img}}
$$

Esto fuerza una interacción directa entre dimensiones alineadas.

#### **4. Variante bilineal de baja dimensión**

Una versión simplificada de interacción bilineal:

1. Proyecta ambas modalidades a una dimensión común baja:
   $$
   \tilde z_{\text{text}} = W_t z_{\text{text}}, \quad \tilde z_{\text{img}} = W_i z_{\text{img}}
   $$
2. Combina mediante producto elemento a elemento:
   $$
   z = \tilde z_{\text{text}} \odot \tilde z_{\text{img}}
   $$

Esto aproxima una interacción multiplicativa más amplia que la concatenación, pero con menor costo que una capa bilineal completa.

#### **Tarea**

Implementa las cuatro variantes y entrenar un clasificador comparable sobre cada una.

#### **Condición importante**

Para que la comparación sea justa:

* mantiene el mismo split de datos,
* usa embeddings base idénticos,
* usa el mismo criterio de evaluación,
* y controla, en lo posible, el número de parámetros.

#### **Qué analizar**

* ¿Qué método obtiene mejor desempeño?
* ¿Cuál parece más estable?
* ¿Cuál es más sensible al tamaño del dataset?
* ¿Qué estrategias permiten interacciones entre modalidades y cuáles solo agregan información?.

#### **Entrega esperada**

* Tabla comparativa con accuracy, macro-F1 y número aproximado de parámetros.
* Discusión breve sobre expresividad vs. simplicidad.

#### **D. Extensión de investigación: modalidad no disponible**

En aplicaciones reales, a veces una modalidad no está disponible: puede faltar la imagen, o el texto puede estar incompleto. Este ejercicio estudia la robustez de las estrategias de fusión ante esa situación.

##### **Tarea**

Construye al menos tres escenarios:

1. **Falta texto en una fracción del conjunto**

   * Por ejemplo, reemplaza el texto por cadena vacía o por un embedding nulo en un 20% de los ejemplos.

2. **Falta imagen en otra fracción**

   * Por ejemplo, reemplaza la imagen por un vector de ceros o una imagen negra.

3. **Faltan ambas modalidades en distintos grados**

   * Prueba varios niveles: 10%, 20%, 40%.

#### **Diseño experimental sugerido**

* Eligue 2 o 3 estrategias de fusión para comparar.
* Evalua en condiciones limpias y en condiciones con modalidad faltante.
* Mantiene el entrenamiento y el test controlados para que la comparación sea clara.

#### **Preguntas guía**

* ¿Qué método cae menos cuando falta una modalidad?
* ¿Los modelos de gating reaccionan mejor que la concatenación?
* ¿Conviene entrenar ya con ejemplos incompletos para aumentar robustez?
* ¿Hay una modalidad que el modelo use como "respaldo" de la otra?.

#### **Entrega esperada**

* Gráfico o tabla con desempeño según porcentaje de modalidades faltantes.
* Conclusión sobre robustez.
* Reflexión sobre qué implicancias tendría esto en una aplicación real.

#### **E. Extensión conceptual: comparación entre GMU, FiLM y TFN**

Este ejercicio no requiere programación adicional, sino una explicación conceptual breve y clara.

##### **Tarea**

Escribe una sección corta, en lenguaje propio, respondiendo:

##### **1. ¿Qué añade GMU frente a concatenación?**

Puntos que deberían aparecer:

* La concatenación solo junta las representaciones y deja que una capa posterior aprenda la combinación.
* **GMU** introduce una compuerta aprendida que decide cuánto aporta cada modalidad en cada ejemplo.
* Esto permite una fusión **adaptativa**, en vez de una combinación fija.

##### **2. ¿Qué añade FiLM frente a GMU?**

Puntos que deberían aparecer:

* GMU decide cómo mezclar modalidades.
* **FiLM** permite que una modalidad **module** a la otra mediante parámetros de escalamiento y desplazamiento.
* No solo mezcla, sino que **condiciona** el procesamiento de una modalidad usando información de la otra.
* Es útil cuando una modalidad debe guiar la interpretación de la otra.

##### **3. ¿Qué añade TFN frente a concat + MLP?**

Puntos que deberían aparecer:

* Concat + MLP puede aprender interacciones, pero de manera implícita.
* **TFN** modela interacciones unimodales, bimodales y trimodales de forma explícita mediante productos tensoriales.
* Esto aumenta la capacidad de capturar relaciones complejas, aunque con mayor costo computacional.

##### **Entrega esperada**

Un texto de entre 1 y 2 párrafos por método comparado, donde se explique:

* qué mecanismo introduce,
* qué ventaja potencial ofrece,
* y qué costo o limitación tiene.



In [ ]:
## Tus respuestas